In [18]:
import kagglehub
from sklearn.model_selection import TimeSeriesSplit
import os

# Download latest version
path = kagglehub.dataset_download("vishardmehta/delhi-pollution-aqi-dataset")

print("Path to dataset files:", path)

print(f"downlaoded files : {os.listdir(path)}")

Path to dataset files: /home/ashish-d/.cache/kagglehub/datasets/vishardmehta/delhi-pollution-aqi-dataset/versions/1
downlaoded files : ['delhi_ncr_aqi_dataset.csv']


In [19]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from sklearn.preprocessing import RobustScaler, LabelEncoder
import lightgbm as lgb
import os
from sklearn.model_selection import train_test_split
import secrets
# --- ENVIRONMENT & PATHS ---
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BASE_PATH = './dataset/public/'
SAVE_PATH = './working/'

os.makedirs(BASE_PATH , exist_ok  = True)
os.makedirs(SAVE_PATH , exist_ok=True)

df = pd.read_csv(os.path.join(path , 'delhi_ncr_aqi_dataset.csv'))

train , test = train_test_split(df , test_size= 0.3 , shuffle=False)

test['id'] = [secrets.token_hex(32) for _ in range(len(test))]

train.to_csv(os.path.join(BASE_PATH , "train.csv") , index = False)
test.to_csv(os.path.join(BASE_PATH , "test.csv") , index = False)

print("Created train and test files")

Created train and test files


In [20]:
# --- DATA LOADING ---
train_df = pd.read_csv(os.path.join(BASE_PATH, 'train.csv'))
test_df = pd.read_csv(os.path.join(BASE_PATH, 'test.csv'))

import joblib
from sklearn.preprocessing import LabelEncoder
import numpy as np

# --- REFINED FEATURE ENGINEERING ---
# (Your build_features function remains exactly the same)
def build_features(df):
    df = df.copy()
    df['h_sin'] = np.sin(2 * np.pi * df['hour'] / 24)
    df['h_cos'] = np.cos(2 * np.pi * df['hour'] / 24)
    df['m_sin'] = np.sin(2 * np.pi * df['month'] / 12)
    df['m_cos'] = np.cos(2 * np.pi * df['month'] / 12)
    
    df['is_monsoon'] = (df['season'] == 'monsoon').astype(int)
    df['is_winter_saturation'] = ((df['season'] == 'winter') & (df['humidity'] > 85)).astype(int)
    
    df['geo_interaction'] = df['latitude'] * df['longitude']
    return df

train_df = build_features(train_df)
test_df = build_features(test_df)

# --- FIXED LABEL ENCODING ---
cat_cols = ['day_of_week', 'season', 'city', 'station']
encoders_dict = {} # Create a dictionary to hold all encoders

for col in cat_cols:
    le = LabelEncoder()
    train_df[col] = le.fit_transform(train_df[col])
    test_df[col] = le.transform(test_df[col])
    encoders_dict[col] = le # Save the fitted encoder into the dictionary

# Scaling Numericals (Including the new engineered flags)
num_cols = ['latitude', 'longitude', 'temperature', 'humidity', 'wind_speed', 'visibility', 
            'h_sin', 'h_cos', 'm_sin', 'm_cos', 'geo_interaction', 'is_monsoon', 'is_winter_saturation']
scaler = RobustScaler()
train_df[num_cols] = scaler.fit_transform(train_df[num_cols])
test_df[num_cols] = scaler.transform(test_df[num_cols])

# --- NEURAL NETWORK (Stage 1) ---
class AstraeaNet(nn.Module):
    def __init__(self, cat_dims, num_dim):
        super().__init__()
        self.embs = nn.ModuleList([nn.Embedding(d, 16) for d in cat_dims])
        self.input_layer = nn.Linear(16 * len(cat_dims) + num_dim, 512)
        
        self.res_block = nn.Sequential(
            nn.Linear(512, 512),
            nn.SiLU(),
            nn.BatchNorm1d(512),
            nn.Dropout(0.2),
            nn.Linear(512, 512),
            nn.SiLU()
        )
        
        self.head = nn.Sequential(
            nn.Linear(512, 256),
            nn.SiLU(),
            nn.Linear(256, 1)
        )

    def forward(self, c, n):
        embeddings = [self.embs[i](c[:, i]) for i in range(len(self.embs))]
        x = torch.cat(embeddings + [n], dim=1)
        x = self.input_layer(x)
        x = x + self.res_block(x)
        return self.head(x)

# --- TRAINING STAGE 1 ---
cat_dims = [train_df[c].nunique() for c in cat_cols]
model = AstraeaNet(cat_dims, len(num_cols)).to(DEVICE)
optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-3)
criterion = nn.HuberLoss(delta=1.5)

train_data = list(zip(train_df[cat_cols].values, train_df[num_cols].values, train_df['aqi'].values))
train_loader = DataLoader(train_data, batch_size=256, shuffle=True)

print("Stage 1: Training Deep Residual Core...")
model.train()
for epoch in range(35):
    for c, n, t in train_loader:
        c, n, t = c.to(DEVICE).long(), n.to(DEVICE).float(), t.to(DEVICE).float().view(-1, 1)
        optimizer.zero_grad()
        loss = criterion(model(c, n), t)
        loss.backward()
        optimizer.step()

# --- RESIDUAL COMPUTATION ---
model.eval()
with torch.no_grad():
    train_c = torch.tensor(train_df[cat_cols].values).to(DEVICE).long()
    train_n = torch.tensor(train_df[num_cols].values).to(DEVICE).float()
    train_preds = model(train_c, train_n).cpu().numpy().flatten()

residuals = train_df['aqi'].values - train_preds

# --- STAGE 2: LIGHTGBM REFINEMENT ---
print("Stage 2: Refining Residuals with LightGBM...")
lgb_params = {
    'n_estimators': 2500,
    'learning_rate': 0.02,
    'num_leaves': 127,
    'max_depth': -1,
    'objective': 'regression',
    'metric': 'rmse',
    'n_jobs': -1
}

res_model = lgb.LGBMRegressor(**lgb_params)
res_model.fit(train_df[cat_cols + num_cols], residuals)

# --- FINAL INFERENCE & SATURATION CALIBRATION ---
with torch.no_grad():
    test_c = torch.tensor(test_df[cat_cols].values).to(DEVICE).long()
    test_n = torch.tensor(test_df[num_cols].values).to(DEVICE).float()
    test_preds_nn = model(test_c, test_n).cpu().numpy().flatten()

test_residuals = res_model.predict(test_df[cat_cols + num_cols])
final_aqi = test_preds_nn + test_residuals

# POST-PROCESSING: Hard Ceiling Calibration
# If the model predicts > 498.5, it is likely the sensor hit saturation (500)
final_aqi = np.where(final_aqi > 498.5, 500.0, final_aqi)
# AQI cannot be negative or lower than the theoretical minimum observed (25)
final_aqi = np.clip(final_aqi, 25.0, 500.0)

# Output
submission = pd.DataFrame({'id': test_df['id'], 'aqi': final_aqi})
submission.to_csv(os.path.join(SAVE_PATH, 'submission.csv'), index=False)
print(f"Final hybrid submission generated at {os.path.join(SAVE_PATH, 'submission.csv')}")

Stage 1: Training Deep Residual Core...
Stage 2: Refining Residuals with LightGBM...
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.001940 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 879
[LightGBM] [Info] Number of data points in the train set: 141164, number of used features: 17
[LightGBM] [Info] Start training from score -1.278723
Final hybrid submission generated at ./working/submission.csv


In [21]:
import joblib

data = {
    "model" : model,
    "res_model": res_model,
    "scaler": scaler,               # Save your scaler!
    "label_encoders": encoders_dict       # Save a dictionary of your encoders!
}
joblib.dump(data , "weights.pkl")

['weights.pkl']